In [ ]:
import mne
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import sklearn
mne.viz.set_browser_backend("qt")
montage=mne.channels.make_standard_montage("standard_1020")

data_folder=Path(r"C:\Users\USER\Desktop\SchizophreniaDataset")
edf_files=sorted(data_folder.glob("*.edf"))
print("Number of edf files: ", len(edf_files))

raw_files={}
for file in edf_files:
    raw_files[file.name]=mne.io.read_raw_edf(file, preload=True, verbose=False)
print("loaded: ", len(raw_files), "recordings")

#labelling the data
labels={}
for key in raw_files.keys():
   if key.startswith("h"):
       labels[key]="healthy"
   elif key.startswith("s"):
       labels[key]="schizophrenic"

#preprocessing of eeg
preprocessed={}
for key, raw in raw_files.items():
    processed=raw.copy()
    processed.filter(l_freq=1.0, h_freq=45.0, verbose=False) #bandwidth selection
    
    processed.set_montage(montage, on_missing="warn") #montage selection
    preprocessed[key]=processed
    
    


Using qt as 2D backend.
Number of edf files:  28
loaded:  28 recordings


In [ ]:
#ICA ANALYSIS
ica_objects={}
for key, raw in preprocessed.items():
   ica=mne.preprocessing.ICA(n_components=19, random_state=42, max_iter="auto")
   ica.fit(raw)
   ica_objects[key]=ica

Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 12.2s.
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 15.0s.
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 10.7s.
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 13.8s.
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 8.9s.
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 11.4s.
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 9.0s.
Fitting ICA to data us

In [ ]:
#ICA FOR EOG ARTIFACT
for subj, ica in ica_objects.items():
    raw=preprocessed[subj].copy()
    
    eog_indices, eog_scores=ica.find_bads_eog(raw, ch_name=['Fp1', 'Fp2'], verbose=False) #identifying optical disturbance by setting proxies
    #muscle_indices, muscle_scores=ica.find_bads_muscle(raw)
    #print(sub, "eog:", list(set(eog_indices)))
    
    ica.exclude=list(eog_indices)
    
cleaned={}
#cleaning the eeg of optically disturbed channels
for subj, ica in ica_objects.items():
    raw=preprocessed[subj].copy()
    ica.apply(raw)
    cleaned[subj]=raw
    raw.save(f'C:/Users/USER/Desktop/schizophrenia/cleaned/{subj.replace(".edf", "")}cleaned_raw.fif', overwrite=True)

Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 1 ICA component
    Projecting back using 19 PCA components
Writing C:\Users\USER\Desktop\schizophrenia\cleaned\h01cleaned_raw.fif
Closing C:\Users\USER\Desktop\schizophrenia\cleaned\h01cleaned_raw.fif
[done]
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 2 ICA components
    Projecting back using 19 PCA components
Writing C:\Users\USER\Desktop\schizophrenia\cleaned\h02cleaned_raw.fif
Closing C:\Users\USER\Desktop\schizophrenia\cleaned\h02cleaned_raw.fif
[done]
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 2 ICA components
    Projecting back using 19 PCA components
Writing C:\Users\USER\Desktop\schizophrenia\cleaned\h03cleaned_raw.fif
Closing C:\Users\USER\Desktop\schizophrenia\cleaned\h03cleaned_raw.fif
[done]
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 1 ICA c

In [ ]:
cleaned_dir=Path(r"C:\Users\USER\Desktop\schizophrenia\cleaned")
files=sorted(cleaned_dir.glob("*.fif"))
#print("Number of files:", len(files))
clean={}
for file in files:
    clean[file.name]=mne.io.read_raw_fif(file, preload=True, verbose=False)
    

Loaded:  28


In [ ]:
#EPOCH EXTRACTION AND REJECTION
import mne
from autoreject import AutoReject
import numpy as np

epochs_dict_clean={}
spatial_brain_matrices={}

for subj, raw in clean.items():
    epochs=mne.make_fixed_length_epochs(raw, duration=2.0, overlap=0.0, preload=True) #for making epochs
    
    ar=AutoReject(random_state=42)
    epochs_clean, reject_log=ar.fit_transform(epochs, return_log=True)
    epochs_dict_clean[subj]=epochs_clean
    print(subj, f"{reject_log.bad_epochs.sum()}/{len(reject_log.bad_epochs)}")
    

In [ ]:
#SAVING THE EPOCHS
import os
epoch={}
epochs_dir='C:/Users/USER/Desktop/schizophrenia/epochs'
for subj, epochs in epochs_dict_clean.items():
    fname = subj.replace('.edf', '') + '_epo.fif'
    epochs.save(os.path.join(epochs_dir, fname), overwrite=True)
    print(f"saved {subj}")

saved h01cleaned_raw.fif
saved h02cleaned_raw.fif
saved h03cleaned_raw.fif


C:\Users\USER\AppData\Local\Temp\ipykernel_15368\3992640706.py:6: RuntimeWarning: Saving epochs with no data
  epochs.save(os.path.join(epochs_dir, fname), overwrite=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_15368\3992640706.py:6: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  epochs.save(os.path.join(epochs_dir, fname), overwrite=True)


saved h04cleaned_raw.fif
saved h05cleaned_raw.fif
saved h06cleaned_raw.fif
saved h07cleaned_raw.fif
saved h08cleaned_raw.fif


C:\Users\USER\AppData\Local\Temp\ipykernel_15368\3992640706.py:6: RuntimeWarning: Saving epochs with no data
  epochs.save(os.path.join(epochs_dir, fname), overwrite=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_15368\3992640706.py:6: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  epochs.save(os.path.join(epochs_dir, fname), overwrite=True)


saved h09cleaned_raw.fif
saved h10cleaned_raw.fif
saved h11cleaned_raw.fif
saved h12cleaned_raw.fif
saved h13cleaned_raw.fif
saved h14cleaned_raw.fif
saved s01cleaned_raw.fif
saved s02cleaned_raw.fif
saved s03cleaned_raw.fif
saved s04cleaned_raw.fif
saved s05cleaned_raw.fif
saved s06cleaned_raw.fif
saved s07cleaned_raw.fif
saved s08cleaned_raw.fif
saved s09cleaned_raw.fif
saved s10cleaned_raw.fif


C:\Users\USER\AppData\Local\Temp\ipykernel_15368\3992640706.py:6: RuntimeWarning: Saving epochs with no data
  epochs.save(os.path.join(epochs_dir, fname), overwrite=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_15368\3992640706.py:6: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  epochs.save(os.path.join(epochs_dir, fname), overwrite=True)


saved s11cleaned_raw.fif
saved s12cleaned_raw.fif
saved s13cleaned_raw.fif
saved s14cleaned_raw.fif


In [ ]:
ica_objects['h03.edf'].exclude = [c for c in ica_objects['h03.edf'].exclude if c != 0]
ica_objects['h08.edf'].exclude = [c for c in ica_objects['h08.edf'].exclude if c != 0]
ica_objects['s10.edf'].exclude = [c for c in ica_objects['s10.edf'].exclude if c != 0]

# confirm
for subj in ['h03.edf', 'h08.edf', 's10.edf']:
    print(subj, ica_objects[subj].exclude)

for subj in ['h03.edf', 'h08.edf', 's10.edf']:
    raw = preprocessed[subj].copy()
    ica_objects[subj].apply(raw)
    cleaned[subj] = raw
    fname = f'C:/Users/USER/Desktop/schizophrenia/cleaned/{subj.replace(".edf","")}cleaned_raw.fif'
    raw.save(fname, overwrite=True)
    print(f"re-cleaned and saved {subj}")
    

h03.edf [np.int64(6)]
h08.edf []
s10.edf []
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 1 ICA component
    Projecting back using 19 PCA components
Overwriting existing file.
Writing C:\Users\USER\Desktop\schizophrenia\cleaned\h03cleaned_raw.fif
Overwriting existing file.
Closing C:\Users\USER\Desktop\schizophrenia\cleaned\h03cleaned_raw.fif
[done]
re-cleaned and saved h03.edf
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 0 ICA components
    Projecting back using 19 PCA components
Overwriting existing file.
Writing C:\Users\USER\Desktop\schizophrenia\cleaned\h08cleaned_raw.fif
Overwriting existing file.
Closing C:\Users\USER\Desktop\schizophrenia\cleaned\h08cleaned_raw.fif
[done]
re-cleaned and saved h08.edf
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 0 ICA components
    Projecting back using 19 PCA components
Overwriting existing file.
Writing C:

In [ ]:
#EPOCH EXTRACTION AND REJECTION OF FAILED SUBJECTS
from autoreject import AutoReject
import mne

for subj in ['h03.edf', 'h08.edf', 's10.edf']:
    raw = cleaned[subj]
    epochs = mne.make_fixed_length_epochs(raw, duration=2.0, overlap=0.0, preload=True)
    
    ar = AutoReject(random_state=42)
    epochs_clean, reject_log = ar.fit_transform(epochs, return_log=True)
    epochs_dict_clean[subj] = epochs_clean
    
    print(subj, f"{reject_log.bad_epochs.sum()}/{len(reject_log.bad_epochs)} epochs rejected, {len(epochs_clean)} surviving")

In [ ]:
#DETECTION OF CORRUPTED TIMESTAMPS OF SCHIZOPHRENIA PATIENT 10
raw = cleaned['s10.edf']
data = raw.get_data()
import numpy as np

# Find timepoints where signal amplitude is extreme across many channels at once
threshold = np.percentile(np.abs(data), 98.5)  # top 0.5% amplitude as a rough cutoff
extreme_mask = (np.abs(data) > threshold).sum(axis=0) > 10  # >10 of 19 channels extreme at once

times = raw.times
bad_times = times[extreme_mask]

if len(bad_times) > 0:
    # group consecutive bad timepoints into intervals
    gaps = np.where(np.diff(bad_times) > 1.0)[0]
    starts = [bad_times[0]] + [bad_times[g+1] for g in gaps]
    ends = [bad_times[g] for g in gaps] + [bad_times[-1]]
    print("Detected bad intervals (s):")
    for s, e in zip(starts, ends):
        print(f"  {s:.1f} - {e:.1f}  (duration: {e-s:.1f}s)")
    print(f"\nTotal contaminated: {sum(e-s for s,e in zip(starts,ends)):.1f}s out of {times[-1]:.1f}s total")
else:
    print("No extreme multi-channel bursts detected at this threshold")

Detected bad intervals (s):
  123.6 - 123.6  (duration: 0.0s)
  207.3 - 207.3  (duration: 0.0s)
  210.7 - 210.7  (duration: 0.0s)
  271.1 - 274.0  (duration: 2.9s)
  275.8 - 276.9  (duration: 1.1s)
  307.3 - 307.8  (duration: 0.4s)
  589.0 - 589.6  (duration: 0.6s)

Total contaminated: 5.0s out of 850.0s total


In [ ]:
#UPDATING EPOCH FILES AFTER CORRECTING
import os

epochs_dir = 'C:/Users/USER/Desktop/schizophrenia/epochs'

for subj in ['h03.edf', 'h08.edf']:
    fname = subj.replace('.edf', '') + '_epo.fif'
    epochs_dict_clean[subj].save(os.path.join(epochs_dir, fname), overwrite=True)
    print(f"saved corrected {subj}")

saved corrected h03.edf
saved corrected h08.edf


In [49]:
import os

epochs_dir = 'C:/Users/USER/Desktop/schizophrenia/epochs'

for fname in os.listdir(epochs_dir):
    if fname.endswith('_epo.fif'):
        epochs_temp = mne.read_epochs(os.path.join(epochs_dir, fname), preload=True, verbose=False)
        print(fname, len(epochs_temp))

h01cleaned_raw.fif_epo.fif 453
h02cleaned_raw.fif_epo.fif 453
h03cleaned_raw.fif_epo.fif 399
h04cleaned_raw.fif_epo.fif 445
h05cleaned_raw.fif_epo.fif 415
h06cleaned_raw.fif_epo.fif 459
h07cleaned_raw.fif_epo.fif 427
h08cleaned_raw.fif_epo.fif 374
h09cleaned_raw.fif_epo.fif 449
h10cleaned_raw.fif_epo.fif 557
h11cleaned_raw.fif_epo.fif 409
h12cleaned_raw.fif_epo.fif 414
h13cleaned_raw.fif_epo.fif 482
h14cleaned_raw.fif_epo.fif 406
s01cleaned_raw.fif_epo.fif 413
s02cleaned_raw.fif_epo.fif 472
s03cleaned_raw.fif_epo.fif 482
s04cleaned_raw.fif_epo.fif 599
s05cleaned_raw.fif_epo.fif 440
s06cleaned_raw.fif_epo.fif 273
s07cleaned_raw.fif_epo.fif 473
s08cleaned_raw.fif_epo.fif 452
s09cleaned_raw.fif_epo.fif 579
s10cleaned_raw.fif_epo.fif 0
s11cleaned_raw.fif_epo.fif 676
s12cleaned_raw.fif_epo.fif 517
s13cleaned_raw.fif_epo.fif 538
s14cleaned_raw.fif_epo.fif 817


In [12]:
#PERFORMING PEARSON CORRELATION OF EACH SUBJECT
import mne
import numpy as np
import os

epochs_dir = 'C:/Users/USER/Desktop/schizophrenia/epochs'
spatial_brain_matrices = {}

for fname in os.listdir(epochs_dir):
    if fname.endswith('_epo.fif'):
        epochs = mne.read_epochs(os.path.join(epochs_dir, fname), preload=True)
        
        if len(epochs) == 0:
            print(f"Skipping {fname} — 0 epochs")
            continue
        
        data = epochs.get_data()  # (n_epochs, 19, n_times)
        n_epochs = data.shape[0]
        
        subj_matrices = np.zeros((n_epochs, 19, 19))
        for i in range(n_epochs):
            subj_matrices[i] = np.corrcoef(data[i])
        
        spatial_brain_matrices[fname] = subj_matrices
        print(fname, subj_matrices.shape)

Reading C:\Users\USER\Desktop\schizophrenia\epochs\h01cleaned_raw.fif_epo.fif ...
    Found the data of interest:
        t =       0.00 ...    1996.00 ms
        0 CTF compensation matrices available
Not setting metadata
453 matching events found
No baseline correction applied
0 projection items activated
h01cleaned_raw.fif_epo.fif (453, 19, 19)
Reading C:\Users\USER\Desktop\schizophrenia\epochs\h02cleaned_raw.fif_epo.fif ...
    Found the data of interest:
        t =       0.00 ...    1996.00 ms
        0 CTF compensation matrices available
Not setting metadata
453 matching events found
No baseline correction applied
0 projection items activated
h02cleaned_raw.fif_epo.fif (453, 19, 19)
Reading C:\Users\USER\Desktop\schizophrenia\epochs\h03cleaned_raw.fif_epo.fif ...
    Found the data of interest:
        t =       0.00 ...    1996.00 ms
        0 CTF compensation matrices available
Not setting metadata
399 matching events found
No baseline correction applied
0 projection items acti

c:\Users\USER\.conda\envs\jupyterlab-env\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


Not setting metadata
445 matching events found
No baseline correction applied
0 projection items activated
h04cleaned_raw.fif_epo.fif (445, 19, 19)
Reading C:\Users\USER\Desktop\schizophrenia\epochs\h05cleaned_raw.fif_epo.fif ...
    Found the data of interest:
        t =       0.00 ...    1996.00 ms
        0 CTF compensation matrices available
Not setting metadata
415 matching events found
No baseline correction applied
0 projection items activated
h05cleaned_raw.fif_epo.fif (415, 19, 19)
Reading C:\Users\USER\Desktop\schizophrenia\epochs\h06cleaned_raw.fif_epo.fif ...
    Found the data of interest:
        t =       0.00 ...    1996.00 ms
        0 CTF compensation matrices available
Not setting metadata
459 matching events found
No baseline correction applied
0 projection items activated
h06cleaned_raw.fif_epo.fif (459, 19, 19)
Reading C:\Users\USER\Desktop\schizophrenia\epochs\h07cleaned_raw.fif_epo.fif ...
    Found the data of interest:
        t =       0.00 ...    1996.00 ms

C:\Users\USER\AppData\Local\Temp\ipykernel_6288\3866684690.py:11: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  epochs = mne.read_epochs(os.path.join(epochs_dir, fname), preload=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_6288\3866684690.py:11: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  epochs = mne.read_epochs(os.path.join(epochs_dir, fname), preload=True)


    Found the data of interest:
        t =       0.00 ...    1996.00 ms
        0 CTF compensation matrices available
Not setting metadata
676 matching events found
No baseline correction applied
0 projection items activated
s11cleaned_raw.fif_epo.fif (676, 19, 19)
Reading C:\Users\USER\Desktop\schizophrenia\epochs\s12cleaned_raw.fif_epo.fif ...
    Found the data of interest:
        t =       0.00 ...    1996.00 ms
        0 CTF compensation matrices available
Not setting metadata
517 matching events found
No baseline correction applied
0 projection items activated
s12cleaned_raw.fif_epo.fif (517, 19, 19)
Reading C:\Users\USER\Desktop\schizophrenia\epochs\s13cleaned_raw.fif_epo.fif ...
    Found the data of interest:
        t =       0.00 ...    1996.00 ms
        0 CTF compensation matrices available
Not setting metadata
538 matching events found
No baseline correction applied
0 projection items activated
s13cleaned_raw.fif_epo.fif (538, 19, 19)
Reading C:\Users\USER\Desktop\schiz

In [13]:
import pickle
with open('C:/Users/USER/Desktop/schizophrenia/spatial_brain_matrices.pkl', 'wb') as f:
    pickle.dump(spatial_brain_matrices, f)

In [15]:
spatial_brain_matrices_reordered = {}
for subj, matrices in spatial_brain_matrices.items():
    # matrices is (n_epochs, 19, 19) → transpose to (19, 19, n_epochs)
    spatial_brain_matrices_reordered[subj] = np.transpose(matrices, (1, 2, 0))
    print(subj, spatial_brain_matrices_reordered[subj].shape)

h01cleaned_raw.fif_epo.fif (19, 19, 453)
h02cleaned_raw.fif_epo.fif (19, 19, 453)
h03cleaned_raw.fif_epo.fif (19, 19, 399)
h04cleaned_raw.fif_epo.fif (19, 19, 445)
h05cleaned_raw.fif_epo.fif (19, 19, 415)
h06cleaned_raw.fif_epo.fif (19, 19, 459)
h07cleaned_raw.fif_epo.fif (19, 19, 427)
h08cleaned_raw.fif_epo.fif (19, 19, 374)
h09cleaned_raw.fif_epo.fif (19, 19, 449)
h10cleaned_raw.fif_epo.fif (19, 19, 557)
h11cleaned_raw.fif_epo.fif (19, 19, 409)
h12cleaned_raw.fif_epo.fif (19, 19, 414)
h13cleaned_raw.fif_epo.fif (19, 19, 482)
h14cleaned_raw.fif_epo.fif (19, 19, 406)
s01cleaned_raw.fif_epo.fif (19, 19, 413)
s02cleaned_raw.fif_epo.fif (19, 19, 472)
s03cleaned_raw.fif_epo.fif (19, 19, 482)
s04cleaned_raw.fif_epo.fif (19, 19, 599)
s05cleaned_raw.fif_epo.fif (19, 19, 440)
s06cleaned_raw.fif_epo.fif (19, 19, 273)
s07cleaned_raw.fif_epo.fif (19, 19, 473)
s08cleaned_raw.fif_epo.fif (19, 19, 452)
s09cleaned_raw.fif_epo.fif (19, 19, 579)
s11cleaned_raw.fif_epo.fif (19, 19, 676)
s12cleaned_raw.f

In [16]:
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Labels per subject
subjects = list(spatial_brain_matrices.keys())
labels_by_subject = {s: (1 if s.startswith('s') else 0) for s in subjects}

# 2. Split at SUBJECT level, stratified
train_subjects, test_subjects = train_test_split(
    subjects,
    test_size=3/7,
    stratify=[labels_by_subject[s] for s in subjects],
    random_state=42
)

print("Train subjects:", len(train_subjects))
print("Test subjects:", len(test_subjects))

# 3. Build epoch-level X/y from only the subjects in each split
def build_dataset(subject_list, matrices_dict, labels_dict):
    X, y = [], []
    for subj in subject_list:
        mats = matrices_dict[subj]  # (n_epochs, 19, 19)
        for i in range(mats.shape[0]):
            X.append(mats[i, :, :])
            y.append(labels_dict[subj])
    return np.array(X), np.array(y)

X_train, y_train = build_dataset(train_subjects, spatial_brain_matrices, labels_by_subject)
X_test, y_test = build_dataset(test_subjects, spatial_brain_matrices, labels_by_subject)

# 4. Add channel dimension for CNN input
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

Train subjects: 15
Test subjects: 12
(7461, 19, 19, 1) (7461,)
(5412, 19, 19, 1) (5412,)


In [17]:
import numpy as np
print("Train class balance:", np.bincount(y_train))
print("Test class balance:", np.bincount(y_test))

Train class balance: [3666 3795]
Test class balance: [2476 2936]


In [18]:
print('X_train' in dir())
print('spatial_brain_matrices' in dir())

True
True


In [ ]:
#adjusting NAN epoch 398th in h03
def build_dataset(subject_list, matrices_dict, labels_dict):
    X, y = [], []
    for subj in subject_list:
        mats = matrices_dict[subj]
        for i in range(mats.shape[0]):
            if not np.isnan(mats[i]).any():
                X.append(mats[i, :, :])
                y.append(labels_dict[subj])
    return np.array(X), np.array(y)

X_train, y_train = build_dataset(train_subjects, spatial_brain_matrices, labels_by_subject)
X_test, y_test = build_dataset(test_subjects, spatial_brain_matrices, labels_by_subject)
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("X_train has NaN:", np.isnan(X_train).any())
print("X_test has NaN:", np.isnan(X_test).any())
print(X_train.shape, X_test.shape)

X_train has NaN: False
X_test has NaN: False
(7460, 19, 19, 1) (5412, 19, 19, 1)


In [ ]:
#building the cnn network
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Convert your numpy arrays to tensors
# PyTorch expects (N, C, H, W) — channel first, unlike TF's channel-last
X_train_t = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)  # (N, 1, 19, 19)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

X_test_t = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(2)
        
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 4 * 4, 32)  # 19->9->4 after two pools
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(32, 1)
        
    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.sigmoid(self.fc2(x))
        return x

model = SimpleCNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [25]:
#training the cnn model
epochs = 20
for epoch in range(epochs):
    model.train()
    train_loss, correct, total = 0, 0, 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * xb.size(0)
        correct += ((preds > 0.5).float() == yb).sum().item()
        total += yb.size(0)
    
    train_acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} — loss: {train_loss/total:.4f} — acc: {train_acc:.4f}")

Epoch 1/20 — loss: 0.2017 — acc: 0.9208
Epoch 2/20 — loss: 0.0643 — acc: 0.9811
Epoch 3/20 — loss: 0.0405 — acc: 0.9885
Epoch 4/20 — loss: 0.0298 — acc: 0.9913
Epoch 5/20 — loss: 0.0294 — acc: 0.9920
Epoch 6/20 — loss: 0.0282 — acc: 0.9909
Epoch 7/20 — loss: 0.0190 — acc: 0.9948
Epoch 8/20 — loss: 0.0189 — acc: 0.9946
Epoch 9/20 — loss: 0.0135 — acc: 0.9965
Epoch 10/20 — loss: 0.0140 — acc: 0.9945
Epoch 11/20 — loss: 0.0194 — acc: 0.9930
Epoch 12/20 — loss: 0.0131 — acc: 0.9962
Epoch 13/20 — loss: 0.0114 — acc: 0.9962
Epoch 14/20 — loss: 0.0125 — acc: 0.9968
Epoch 15/20 — loss: 0.0084 — acc: 0.9966
Epoch 16/20 — loss: 0.0082 — acc: 0.9972
Epoch 17/20 — loss: 0.0077 — acc: 0.9977
Epoch 18/20 — loss: 0.0044 — acc: 0.9984
Epoch 19/20 — loss: 0.0117 — acc: 0.9962
Epoch 20/20 — loss: 0.0116 — acc: 0.9953


In [26]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb)
        correct += ((preds > 0.5).float() == yb).sum().item()
        total += yb.size(0)

print(f"Test accuracy: {correct/total:.4f}")

Test accuracy: 0.4837


In [27]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# ============================================================
# PART 1: Classical ML baselines
# ============================================================
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

scaler = StandardScaler()
X_train_flat_scaled = scaler.fit_transform(X_train_flat)
X_test_flat_scaled = scaler.transform(X_test_flat)

print("=== Classical ML baselines ===")
for name, clf in [
    ('SVM', SVC()),
    ('LogReg', LogisticRegression(max_iter=2000)),
    ('RandomForest', RandomForestClassifier(n_estimators=200, random_state=42))
]:
    clf.fit(X_train_flat_scaled, y_train)
    train_acc = clf.score(X_train_flat_scaled, y_train)
    test_acc = clf.score(X_test_flat_scaled, y_test)
    print(f"{name}: train={train_acc:.3f}, test={test_acc:.3f}")

# ============================================================
# PART 2: Regularized CNN with early stopping
# ============================================================
X_train_t = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=32, shuffle=False)


class RegularizedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        self.pool1 = nn.MaxPool2d(2)
        self.dropout1 = nn.Dropout(0.5)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(8 * 9 * 9, 8)
        self.dropout2 = nn.Dropout(0.6)
        self.fc2 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.dropout1(x)
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout2(x)
        x = torch.sigmoid(self.fc2(x))
        return x

model = RegularizedCNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.01)

def evaluate(loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb)
            loss_sum += criterion(preds, yb).item() * xb.size(0)
            correct += ((preds > 0.5).float() == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

print("\n=== CNN training (with early stopping) ===")
best_test_acc = 0
patience, patience_counter = 7, 0
best_state = None

for epoch in range(50):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

    train_loss, train_acc = evaluate(train_loader)
    test_loss, test_acc = evaluate(test_loader)
    print(f"Epoch {epoch+1} — train_acc: {train_acc:.4f} — test_acc: {test_acc:.4f}")

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(best_state)
print(f"\nBest test accuracy: {best_test_acc:.4f}")

# ============================================================
# PART 3: Per-subject majority vote (most meaningful metric)
# ============================================================
print("\n=== Per-subject evaluation ===")
model.eval()
subject_results = {}
with torch.no_grad():
    for subj in test_subjects:
        mats = spatial_brain_matrices[subj]
        valid_epochs = [mats[i] for i in range(mats.shape[0]) if not np.isnan(mats[i]).any()]
        x = torch.tensor(np.array(valid_epochs), dtype=torch.float32).unsqueeze(1)
        preds = model(x)
        majority_pred = (preds.mean() > 0.5).item()
        true_label = labels_by_subject[subj]
        subject_results[subj] = (majority_pred, true_label)
        print(subj, "predicted:", int(majority_pred), "actual:", true_label)

correct_subjects = sum(1 for pred, true in subject_results.values() if pred == true)
print(f"\nPer-subject accuracy: {correct_subjects}/{len(subject_results)} = {correct_subjects/len(subject_results):.3f}")

=== Classical ML baselines ===
SVM: train=0.998, test=0.509
LogReg: train=0.997, test=0.433
RandomForest: train=1.000, test=0.484

=== CNN training (with early stopping) ===
Epoch 1 — train_acc: 0.8660 — test_acc: 0.5161
Epoch 2 — train_acc: 0.9155 — test_acc: 0.5492
Epoch 3 — train_acc: 0.9287 — test_acc: 0.5549
Epoch 4 — train_acc: 0.9566 — test_acc: 0.5394
Epoch 5 — train_acc: 0.9619 — test_acc: 0.5342
Epoch 6 — train_acc: 0.9605 — test_acc: 0.5530
Epoch 7 — train_acc: 0.9721 — test_acc: 0.5488
Epoch 8 — train_acc: 0.9731 — test_acc: 0.5514
Epoch 9 — train_acc: 0.9749 — test_acc: 0.5325
Epoch 10 — train_acc: 0.9755 — test_acc: 0.5432
Early stopping at epoch 10

Best test accuracy: 0.5549

=== Per-subject evaluation ===
h07cleaned_raw.fif_epo.fif predicted: 0 actual: 0
h08cleaned_raw.fif_epo.fif predicted: 1 actual: 0
s14cleaned_raw.fif_epo.fif predicted: 1 actual: 1
s05cleaned_raw.fif_epo.fif predicted: 1 actual: 1
s03cleaned_raw.fif_epo.fif predicted: 1 actual: 1
h14cleaned_raw.fif